# Project 1: Handwritten Digit Recognition (MNIST)

This notebook implements a neural network to classify handwritten digits from the MNIST dataset.

## Objectives
- Load and preprocess the MNIST dataset
- Build a feedforward neural network
- Train the model to classify digits 0-9
- Evaluate and visualize results

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import sys
import os
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'src'))
from deep_learning.neural_networks import SimpleNeuralNetwork

print("Libraries imported successfully!")

## Step 1: Load and Preprocess Data

In [ ]:
# Load MNIST dataset
print("Loading MNIST dataset...")
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
X, y = mnist.data, mnist.target.astype(int)

print(f"Dataset shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Unique labels: {np.unique(y)}")

# Normalize pixel values to [0, 1]
X = X / 255.0

# Visualize sample digits
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i in range(10):
    idx = np.where(y == i)[0][0]
    axes[i//5, i%5].imshow(X[idx].reshape(28, 28), cmap='gray')
    axes[i//5, i%5].set_title(f'Digit: {y[idx]}')
    axes[i//5, i%5].axis('off')
plt.suptitle('Sample MNIST Digits', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 2: Prepare Data for Training

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

# One-hot encode labels
def one_hot_encode(y, num_classes=10):
    encoded = np.zeros((len(y), num_classes))
    encoded[np.arange(len(y)), y] = 1
    return encoded

y_train_encoded = one_hot_encode(y_train)
y_test_encoded = one_hot_encode(y_test)

print(f"Encoded labels shape: {y_train_encoded.shape}")

## Step 3: Create and Train Neural Network

In [ ]:
# Create neural network: 784 input -> 128 -> 64 -> 10 output
print("Creating neural network...")
nn = SimpleNeuralNetwork(layers=[784, 128, 64, 10], learning_rate=0.01)

print("Training neural network...")
loss_history = nn.train(X_train, y_train_encoded, epochs=50, verbose=True)

## Step 4: Evaluate Model

In [ ]:
# Make predictions
predictions = nn.predict(X_test)
predicted_classes = np.argmax(predictions, axis=1)
accuracy = np.mean(predicted_classes == y_test)

print(f"Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

# Confusion matrix
from sklearn.metrics import confusion_matrix, classification_report
cm = confusion_matrix(y_test, predicted_classes)

print("\nConfusion Matrix:")
print(cm)
print("\nClassification Report:")
print(classification_report(y_test, predicted_classes))

## Step 5: Visualize Results

In [ ]:
# Plot training loss
plt.figure(figsize=(10, 5))
plt.plot(loss_history)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss - MNIST Classification')
plt.grid(True, alpha=0.3)
plt.show()

# Visualize sample predictions
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i in range(10):
    idx = np.where(y_test == i)[0][0]
    axes[i//5, i%5].imshow(X_test[idx].reshape(28, 28), cmap='gray')
    pred_prob = predictions[idx][predicted_classes[idx]]
    axes[i//5, i%5].set_title(f'True: {y_test[idx]}\nPred: {predicted_classes[idx]}\nProb: {pred_prob:.2f}')
    axes[i//5, i%5].axis('off')
plt.suptitle('Sample Predictions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Visualize confusion matrix
plt.figure(figsize=(10, 8))
plt.imshow(cm, cmap='Blues', interpolation='nearest')
plt.colorbar()
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
for i in range(10):
    for j in range(10):
        plt.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=8)
plt.xticks(range(10))
plt.yticks(range(10))
plt.tight_layout()
plt.show()